In [105]:
from gensim.models import Word2Vec
import pandas as pd
import pronto
import pickle

with open('/home/peng/PycharmProjects/featurematch/data/processed/final/ids/mondo_id_name.dict', 'rb') as handle:
    mondo_id_name = pickle.load(handle)
    
with open('/home/peng/PycharmProjects/featurematch/data/processed/final/ids/gene_id_name.dict', 'rb') as handle:
    gene_id_name = pickle.load(handle)

with open('/home/peng/PycharmProjects/featurematch/data/processed/final/ids/hpo_id_name.dict', 'rb') as handle:
    hpo_id_name = pickle.load(handle)
    
patient_disease = {}
patient_gene = {}

evaluation_patients = '/home/peng/PycharmProjects/featurematch/data/processed/patients/reform/evaluation/evaluation_pedia_mapped.tsv'
training_patients = '/home/peng/PycharmProjects/featurematch/data/processed/patients/reform/training/training_mapping.tsv'


with open (training_patients, 'r') as infile:
    content = infile.read().splitlines()
    content = [x.split('\t') for x in content]
    for line in content:
        patient_id = line[0]
        disease = line[1]
        if disease in mondo_id_name:
            disease = mondo_id_name[disease]
        gene = line[3]
        patient_disease[patient_id] = disease 
        patient_gene[patient_id] = gene
        

model = Word2Vec.load('../../../model/with_patients/with_patients.model')
list_evaluation = list()

with open(evaluation_patients, 'r') as e_file:
    content = e_file.read().splitlines()
    content = [x.split('\t') for x in content]
    for line in content:
        patient_id = line[0]
        disease = mondo_id_name[line[1]]
        features = line[2].split(',')
        gene = gene_id_name[line[3]]
        if line[4].split(',') == ['unknown']:
            absent_features = []
        else:
            absent_features = line[4].split(',')

        similar_nodes = model.most_similar(positive=features, negative=absent_features, topn=1000)
        similar_nodes_features_filters = []
        
        for node in similar_nodes:
            if node[0].startswith('Patient'):
                if patient_gene[node[0]] in gene_id_name:
                    similar_nodes_features_filters.append(node[0]+ '/' + patient_disease[node[0]]+ '/' + gene_id_name[patient_gene[node[0]]])
                else:
                    similar_nodes_features_filters.append(node[0]+ '/' + patient_disease[node[0]]+ '/' + patient_gene[node[0]])                                   
            elif node[0].startswith('Entrez'):
                 similar_nodes_features_filters.append(gene_id_name[node[0]])
            elif node[0].startswith('MONDO'):
                similar_nodes_features_filters.append(mondo_id_name[node[0]])
        list_evaluation.append([patient_id, disease, gene, ','.join(hpo_id_name[x] for x in features), str(len(features)), ', '.join(similar_nodes_features_filters[:10])])        


/home/peng/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:51: DeprecationWarning: Call to deprecated `most_similar` (Method will be removed in 4.0.0, use self.wv.most_similar() instead).


In [106]:
pd.set_option('display.max_colwidth', -1)
evaluationframe = pd.DataFrame(list_evaluation, columns = ['Id', 'Syndrome', 'Gene', 'Features', 'No.features', 'Prioritization result'])

In [107]:
evaluationframe.to_excel('../../../model/with_patients/evaluation.xlsx')